In [1]:
# Mount Google Drive for persistent storage
from google.colab import drive
print('Mounting Google Drive...')
drive.mount('/content/drive')

# Create a directory for our data
!mkdir -p '/content/drive/MyDrive/isazi_recruitment'
DB_PATH = '/content/drive/MyDrive/isazi_recruitment/recruitment.db'
print(f'Database will be stored at: {DB_PATH}')

ModuleNotFoundError: No module named 'google'

# Run ISAZI Recruitment Dashboard in Google Colab

This notebook clones the repository, installs dependencies (including heavier libs: `prophet`, `shap`, `optuna`, `xgboost`), runs tests, creates sample data, and starts the Streamlit dashboard exposed via ngrok.

Notes:
- Heavy packages (Prophet, SHAP) may take several minutes to install and can require additional system libraries.
- If your repo is private, either provide a GitHub token or upload the repo as a zip.
- For persistent DB between runs, consider mounting Google Drive and writing the SQLite file there.

In [ ]:
# 1) Clone the repo (adjust if using a different path or private repo)
import os
os.chdir('/content')
repo_url = 'https://github.com/olwethusibisi7/dataset.git'
if not os.path.exists('dataset'):
    !git clone {repo_url} dataset
else:
    print('dataset already exists; pulling latest')
    os.chdir('dataset')
    !git pull || true
    os.chdir('..')

In [ ]:
import os

# Navigate to the cloned repo and verify structure
os.chdir('/content/dataset')
print('Current directory: - colab_run_dashboard.ipynb:5', os.getcwd())
print('\nRepository contents: - colab_run_dashboard.ipynb:6')
!ls -la
print('\nChecking for key files: - colab_run_dashboard.ipynb:8')
!ls -la dashboard.py recruitment_functions.py requirements.txt 2>/dev/null || echo 'Some files not found'

In [ ]:
# 2) Install Python packages (this may take a few minutes)
# Adjust the list below if your project requires additional packages.
!python -m pip install --upgrade pip setuptools wheel
!pip install -q streamlit pyngrok pytest pandas numpy scikit-learn plotly xgboost prophet shap optuna
print('Install finished (or still running in background).')

If the `prophet` install fails due to pystan/build errors, try installing `cmdstanpy` and `prophet` separately or use a lighter stack (skip prophet). SHAP may also require a C compiler; Colab usually has one available.

In [ ]:
# 3) (Optional) Provide an ngrok auth token for stable tunnels.
# If you don't have one, leave empty to use an ephemeral tunnel.
NGROK_AUTH_TOKEN = ''  # <-- paste your token here as a string if available
if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print('ngrok auth token configured')
else:
    print('No ngrok token provided; tunnel will be ephemeral (may have limits)')

In [ ]:
# 4) Run tests (adjust folder names if necessary)
import os
os.chdir('/content/dataset')
print('Running pytest for ISAZI DASH/tests (if present) and tests/...')
# Run the ISAZI DASH tests (path contains a space) then fallback to regular tests
!pytest -q 
 || true
!pytest -q tests || true

Create sample data so the dashboard has something to display. The project exposes `create_sample_data()` in `recruitment_functions.py`.

In [ ]:
# 5) Create sample DB data (so Streamlit has rows to show)
import sys, os
os.chdir('/content/dataset')
# Add project root to path so imports work
sys.path.insert(0, os.getcwd())
from recruitment_functions import create_sample_data, load_recruitment_data
create_sample_data(force=True)
df = load_recruitment_data()
print('Loaded rows:', len(df))

In [ ]:
# 6) Start Streamlit and expose via ngrok; prints public URL
import os, subprocess, time
from pyngrok import ngrok
REPO_DIR = '/content/dataset'
PORT = 8501
os.chdir(REPO_DIR)
print('Starting ngrok tunnel...')
tunnel = ngrok.connect(PORT)
print('Public URL:', tunnel.public_url)
logfile = os.path.join(REPO_DIR, 'streamlit_colab.log')
cmd = f'nohup streamlit run dashboard.py --server.port {PORT} --server.headless true > {logfile} 2>&1 &'
print('Starting Streamlit...')
subprocess.call(cmd, shell=True)
time.sleep(3)
print('Streamlit logs (tail):')
!tail -n 40 /content/dataset/streamlit_colab.log || true
print('
Open the public URL above in your browser to view the dashboard.')

Troubleshooting:
- If you see import errors, check the earlier cells' pip install output.
- Long build times for some packages may require patience.
- To persist the SQLite DB between Colab sessions, mount Google Drive and create the DB under `/content/drive/MyDrive/...`.